# The Jedi Council, in a notebook

A runnable, dependency-free reimplementation of the council that keeps
[ilm.red](https://ilm.red)'s machine-built knowledge graph honest.

**Companion reading**

- [Meet the AI Jedi Council — keeping a machine-built knowledge graph honest](https://ilm.red/blog/meet-the-ai-jedi-council-keeping-a-machine-built-knowledge-graph-honest) (Part 1 — this notebook)
- [The Jedi Council, Part 2 — teaching AI to read Urdu aloud](https://ilm.red/blog/the-jedi-council-part-2-teaching-ai-to-read-urdu-aloud) (per-seat models, quorum → high council escalation)

**What you get**

1. The five lens rubrics, verbatim from `public.prompt_registry`.
2. The per-seat dials — weight, temperature, severity — mirroring `public.ai_council_members`.
3. Weighted-mean consensus, and the disagreement spread that a mean hides.
4. Homonym separation by *meaning-in-context* rather than spelling.
5. Four experiments that make the dials visible.
6. Per-club presets, and three tests for whether a preset is wrong.
7. A measured answer to "one model or five?", with recorded results.

**Runs with no API keys.** The default provider is a deterministic offline mock, so every
cell below executes as-is. Set `DEEPSEEK_API_KEY` or `OPENAI_API_KEY` in the environment and
the same code paths call a real model instead. Standard library only — no pip install.

Licensed Apache-2.0, same as the rest of this repository.

## 0 · The Feynman explanation

*Before any of the machinery — what is actually going on, in plain words.*

Imagine you run a library, and a machine reads ten thousand research papers and files every
idea it finds onto a shelf. It is fast, it never gets bored, and it makes one mistake over and
over: **it files by spelling.** Two ideas that happen to share a name land on the same shelf
even when they have nothing to do with each other. That is how a picture of ink spreading
through water ended up on a page about generating images.

So we stopped asking one question — *what is this called?* — and started asking five, each by
a judge who cares about exactly one thing:

- Is it correct, and is it new?
- Does it actually work?
- Could a beginner learn it?
- Will it still matter in five years?
- Is the evidence real, or is it just talk?

Each judge gives a number out of ten. Then we **average** them. Not a vote — an average. A
vote throws away *how sure* everyone was: four judges mildly in favour beat one judge who is
certain it is wrong, and you never learn the disagreement happened. An average keeps it. If
you want one judge to count for more, you make its number worth 1.5 instead of 1. Nobody gets
a veto.

That is the whole idea. Everything below is: which numbers, stored where, and what happens
when you turn them.

### The part we had wrong

For a long time we were not really running five judges. We were asking **one model, in one
breath, to play all five**. That is like asking one person to sit on a jury five times wearing
different hats — you get five votes and one opinion.

When we finally gave each seat its own model and asked them separately, they started
disagreeing with each other. And here is the surprise: **the average barely moved.** What
changed is that we could now *tell* when the average was shaky, because the judges had stopped
pretending to agree.

That turns out to be the useful thing. More judges rarely buy you a better answer. They buy
you a reliable signal about *when not to trust the answer* — and that is worth more, because a
number you know to distrust can be handed to a human, and a confidently wrong number cannot.

Section 8 measures exactly this, on the ink-in-water bug that started it all.

## 1 · Configuration

One dict per seat. These are the shipped production defaults: every lens starts at weight
`1.0` (a uniform prior is the only honest starting point until you have evidence that one
lens predicts club satisfaction better than another), temperature `0.2`, severity `0.5`.

In production these are rows in `public.ai_council_members`, edited through the admin-gated
`set_council_member()` RPC and written to `admin_audit_log`. Here they are a dict you can
edit in place — which is the entire point of the notebook.

In [ ]:
from dataclasses import dataclass, replace, field
from typing import Dict, List, Optional
import json, math, os, hashlib, statistics, urllib.request, urllib.error

@dataclass
class Seat:
    key: str            # lens id, matches prompt_registry key suffix
    persona: str        # the published cast name
    lens: str           # archetype
    rubric: str         # prompt_registry body, verbatim
    weight: float = 1.0       # ai_council_members.weight
    temperature: float = 0.2  # ai_council_members.temperature
    severity: float = 0.5     # ai_council_members.severity  (grading disposition, 0..1)
    enabled: bool = True

# Rubrics copied verbatim from public.prompt_registry, keys council/lens-*, version 1.
COUNCIL: List[Seat] = [
    Seat("theorist",  "Master Yoda", "Theorist",
         "You are the Theorist. Rate this work 0-10 on rigour, novelty and mathematical depth. "
         "Be precise; penalise hand-waving. Return one sentence of rationale."),
    Seat("engineer",  "Anakin Skywalker", "Engineer",
         "You are the Engineer. Rate this work 0-10 on whether it reproduces and whether a "
         "practitioner could use it. Results-first; penalise unavailable code or data."),
    Seat("educator",  "Obi-Wan Kenobi", "Educator",
         "You are the Educator. Rate this work 0-10 on how teachable it is for a motivated club. "
         "Reward clarity and good exposition."),
    Seat("visionary", "Iron Man", "Visionary",
         "You are the Visionary. Rate this work 0-10 on long-term significance rather than "
         "present polish. Reward ideas that open directions."),
    Seat("skeptic",   "Han Solo", "Skeptic",
         "You are the Skeptic. Rate this work 0-10 on TRUST — is the evidence sufficient for the "
         "claims? This is a trust score, not a complaint count. Hunt overclaiming and thin evidence."),
]

CHAIR_RUBRIC = (
    "You are the Chair. Reconcile the members' scores into a weighted-mean consensus and write "
    "one public sentence of rationale. Surface disagreement rather than hiding it. "
    "Never use majority voting."
)

# Gate thresholds, from scripts/classify-term-clubs.mjs.
KEEP_THRESHOLD = 0.5     # relevance >= 0.5 keeps a term in a club
BATCH_SIZE     = 30      # CLASSIFY_BATCH — comparative context per call
MAX_TOKENS     = 8000

for s in COUNCIL:
    print(f"{s.persona:<18} {s.lens:<10} w={s.weight}  temp={s.temperature}  severity={s.severity}")

## 2 · Providers

Two implementations behind one function signature. The mock is deterministic — it hashes the
prompt into a stable pseudo-score — so the notebook is reproducible and free. The real
provider posts to DeepSeek or OpenAI in strict JSON mode, exactly as
`scripts/classify-term-clubs.mjs` does, using `urllib` so there is nothing to install.

In [ ]:
def _stable_unit(*parts: str) -> float:
    """Deterministic float in [0,1) from the given strings — the mock's only source of variety."""
    h = hashlib.sha256("\x1f".join(parts).encode()).digest()
    return int.from_bytes(h[:8], "big") / 2**64

class MockProvider:
    """Offline stand-in. Scores by how much the term's context overlaps the club's focus and
    keywords, saturating rather than clamping so scores stay graded, then applies the seat's
    personality, its severity instruction, and temperature-scaled jitter. Deterministic."""
    name = "mock"
    HALF = 0.15   # overlap at which the mock returns 5/10

    # How each lens departs from the consensus. Real lenses differ far more than this;
    # keeping it modest is what makes the weighting experiment below honest.
    BIAS = {"theorist": 0.8, "engineer": -0.6, "educator": 0.2, "visionary": 1.2, "skeptic": -1.6}

    def complete_json(self, prompt: str, temperature: float, lens: str = "") -> dict:
        focus = _section(prompt, "CLUB FOCUS:")
        term  = _section(prompt, "TERM:")
        ctx   = _section(prompt, "CONTEXT:")
        overlap = _keyword_overlap(f"{term} {ctx}", focus)
        base = 10.0 * overlap / (overlap + self.HALF) if overlap else 0.0

        severity = 0.5
        if "Grade generously" in prompt:   severity = 0.2
        elif "Grade harshly" in prompt:    severity = 0.9
        base += (0.5 - severity) * 4.0      # the disposition, not a multiplier

        jitter = (_stable_unit(prompt, lens, f"{temperature:.2f}") - 0.5) * 6.0 * temperature
        score = max(0.0, min(10.0, base + self.BIAS.get(lens, 0.0) + jitter))
        return {"score": round(score, 2),
                "rationale": f"[mock/{lens}] context-vs-focus overlap {overlap:.2f}"}

class HTTPProvider:
    """DeepSeek (default) or OpenAI, strict JSON mode, 4 retries with backoff — the same
    contract the production jobs use."""
    def __init__(self):
        if os.environ.get("DEEPSEEK_API_KEY"):
            self.name = "deepseek"
            self.url = "https://api.deepseek.com/chat/completions"
            self.key = os.environ["DEEPSEEK_API_KEY"]
            self.model = os.environ.get("CLASSIFY_MODEL", "deepseek-chat")
        elif os.environ.get("OPENAI_API_KEY"):
            self.name = "openai"
            self.url = "https://api.openai.com/v1/chat/completions"
            self.key = os.environ["OPENAI_API_KEY"]
            self.model = os.environ.get("CLASSIFY_MODEL", "gpt-4o-mini")
        else:
            raise RuntimeError("no DEEPSEEK_API_KEY or OPENAI_API_KEY in the environment")

    def complete_json(self, prompt: str, temperature: float, lens: str = "") -> dict:
        body = json.dumps({
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": temperature,
            "max_tokens": 1000,
            "response_format": {"type": "json_object"},
        }).encode()
        req = urllib.request.Request(self.url, data=body, headers={
            "Content-Type": "application/json", "Authorization": f"Bearer {self.key}"})
        last = None
        for attempt in range(4):
            try:
                with urllib.request.urlopen(req, timeout=60) as r:
                    payload = json.loads(r.read())
                return json.loads(payload["choices"][0]["message"]["content"])
            except Exception as e:            # noqa: BLE001 — retry anything transient
                last = e
        raise RuntimeError(f"provider failed after 4 attempts: {last}")

def get_provider():
    try:
        p = HTTPProvider()
        print(f"provider = {p.name} ({p.model})")
        return p
    except RuntimeError as e:
        print(f"provider = mock  ({e}) — set a key to use a real model")
        return MockProvider()

# --- small helpers the mock leans on -------------------------------------------------
_STOP = {"the","a","an","of","in","on","for","to","and","or","is","are","it","its","that",
         "this","with","as","by","from","at","be","not","which","into"}

def _tokens(text: str):
    return [w for w in "".join(c.lower() if c.isalnum() else " " for c in text).split()
            if w not in _STOP and len(w) > 2]

def _keyword_overlap(a: str, b: str) -> float:
    ta, tb = set(_tokens(a)), set(_tokens(b))
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / math.sqrt(len(ta) * len(tb))

def _section(prompt: str, marker: str) -> str:
    if marker not in prompt:
        return ""
    return prompt.split(marker, 1)[1].split("\n\n", 1)[0].strip()

provider = get_provider()

## 3 · Convening the council

One call per seat, each carrying its own rubric, temperature and severity. Then the verdict:

$$\text{consensus} = \frac{\sum_i w_i \cdot s_i}{\sum_i w_i}$$

over the **enabled** seats only. Note what this does *not* do — there is no majority vote and
no veto. A lone `0` against four `9`s still lands at 7.2. If you want a veto you want a
separate gate, not a large weight.

The `spread` returned alongside the mean is the part a single number throws away, and it is
usually the more interesting signal: a 0.62 where every lens said ~0.62 is genuinely
borderline, while a 0.62 where the Skeptic said 0.1 and the Visionary said 0.95 is a homonym
or a mis-tag hiding behind a reasonable average.

In [ ]:
@dataclass
class Verdict:
    term: str
    club: str
    scores: Dict[str, float]          # lens key -> 0..10
    consensus_10: float               # weighted mean, 0..10
    relevance: float                  # consensus_10 / 10 -> 0..1, the stored value
    spread: float                     # max - min across enabled seats
    stdev: float
    kept: bool
    rationales: Dict[str, str] = field(default_factory=dict)

def lens_prompt(seat: Seat, term: str, club: str, focus: str, context: str) -> str:
    harshness = ("Grade generously." if seat.severity < 0.35 else
                 "Grade as the rubric reads." if seat.severity <= 0.65 else
                 "Grade harshly: a 9 must be exceptional.")
    return f"""{seat.rubric}

You are one of five members of the "Jedi Council" of the "{club}" learning club.
Decide whether the concept below genuinely belongs to this club's field.
{harshness}

CLUB FOCUS: {focus}

TERM: {term}

CONTEXT: {context}

Return STRICT JSON only: {{"score": <0-10>, "rationale": "<one sentence>"}}"""

def convene(term: str, club: str, focus: str, context: str,
            council: List[Seat] = None, provider=None) -> Verdict:
    council = council or COUNCIL
    provider = provider or globals()["provider"]
    seats = [s for s in council if s.enabled]
    if not seats:
        raise ValueError("no enabled seats — the council cannot convene")

    scores, rationales = {}, {}
    for s in seats:
        out = provider.complete_json(
            lens_prompt(s, term, club, focus, context), s.temperature, lens=s.key)
        scores[s.key] = float(out.get("score", 0.0))
        rationales[s.key] = str(out.get("rationale", ""))[:200]

    num = sum(s.weight * scores[s.key] for s in seats)
    den = sum(s.weight for s in seats)
    consensus = num / den                       # the whole aggregation, in two lines
    vals = list(scores.values())
    return Verdict(
        term=term, club=club, scores=scores,
        consensus_10=round(consensus, 3),
        relevance=round(consensus / 10.0, 3),
        spread=round(max(vals) - min(vals), 2),
        stdev=round(statistics.pstdev(vals), 2) if len(vals) > 1 else 0.0,
        kept=(consensus / 10.0) >= KEEP_THRESHOLD,
        rationales=rationales,
    )

def show(verdicts: List[Verdict]) -> None:
    keys = [s.key for s in COUNCIL]
    head = f"{'term':<22}{'club':<14}" + "".join(f"{k[:5]:>7}" for k in keys) + f"{'rel':>8}{'spread':>8}  verdict"
    print(head); print("-" * len(head))
    for v in verdicts:
        row = f"{v.term[:21]:<22}{v.club[:13]:<14}"
        row += "".join(f"{v.scores.get(k, float('nan')):>7.1f}" for k in keys)
        row += f"{v.relevance:>8.2f}{v.spread:>8.1f}  {'KEEP' if v.kept else 'drop'}"
        print(row)

## 4 · Fixtures

`diffusion` appears twice on purpose. It is the bug the blog post opens with: the same
spelling, two unrelated meanings, and a knowledge graph that happily merged them and put a
picture of ink in water on an image-generation page.

In [ ]:
CLUBS = {
    "ai": {
        "focus": "Machine learning, deep learning, generative models, and AI systems research.",
        "keywords": ["neural", "network", "transformer", "attention", "embedding", "gradient",
                     "training", "model", "noise", "image", "generative", "denoising", "tokens",
                     "loss", "spectral", "optimisation", "inference", "sequence", "architecture",
                     "curvature"],
    },
    "chemistry": {
        "focus": "Physical chemistry, molecular transport, thermodynamics, and reaction kinetics.",
        "keywords": ["molecules", "concentration", "gradient", "transport", "thermal", "kinetics",
                     "reaction", "entropy", "solvent", "equilibrium", "particles", "motion",
                     "region", "diffusion"],
    },
    "maths": {
        "focus": "Pure and applied mathematics: analysis, algebra, probability, optimisation.",
        "keywords": ["gradient", "optimisation", "minimum", "objective", "differentiable",
                     "algebra", "eigenvalue", "vector", "matrix", "probability", "convergence",
                     "analysis", "theorem", "iterative", "method", "scalar", "linear"],
    },
}

def club_focus(club_id: str) -> str:
    """What the seat is told the club is about — mirrors the club's domains + keywords."""
    c = CLUBS[club_id]
    return c["focus"] + " Keywords: " + ", ".join(c["keywords"]) + "."

FIXTURES = [
    ("diffusion", "ai",
     "A generative model that starts from pure Gaussian noise and iteratively denoises it "
     "into an image, learning the reverse of a forward noising process."),
    ("diffusion", "chemistry",
     "The net transport of molecules from a region of high concentration to low "
     "concentration, driven by the concentration gradient and thermal motion."),
    ("attention", "ai",
     "A learned weighting over a sequence letting a neural network focus on the tokens most "
     "relevant to the current position; the core of the transformer architecture."),
    ("gradient descent", "maths",
     "An iterative first-order optimisation method that steps along the negative gradient of "
     "a differentiable objective to find a local minimum."),
    ("eigenvalue", "ai",
     "A scalar lambda for which a linear map has a vector satisfying Av = lambda v; appears in "
     "spectral methods and in analysing the curvature of loss landscapes."),
    ("transformer", "chemistry",
     "A neural network architecture built from stacked self-attention and feed-forward blocks; "
     "the basis of modern language models."),
    ("the", "ai",
     "A generic English article that was extracted as a candidate term by mistake."),
]

verdicts = [convene(t, c, club_focus(c), ctx) for t, c, ctx in FIXTURES]
show(verdicts)

## 5 · Keeping homonyms apart

Relevance decides *whether* a term belongs to a club. It says nothing about *which sense* of
the term the club means, and that is the harder half of the problem.

The production system embeds a concept's **meaning-in-context** — its definition as used in
this club's papers — rather than its spelling, then compares senses by cosine distance. The
cell below does the same thing with a stdlib hashing vectoriser so it runs anywhere. Swap in
`sentence-transformers` for real embeddings and the structure of the code is unchanged.

In [ ]:
def embed(text: str, dim: int = 512) -> List[float]:
    """Hashing bag-of-words with L2 normalisation — a stand-in for a real sentence encoder.
    Crude, but enough to show that senses separate while spellings do not."""
    v = [0.0] * dim
    for tok in _tokens(text):
        h = int.from_bytes(hashlib.md5(tok.encode()).digest()[:4], "big")
        v[h % dim] += 1.0
    norm = math.sqrt(sum(x * x for x in v)) or 1.0
    return [x / norm for x in v]

def cosine(a: List[float], b: List[float]) -> float:
    return sum(x * y for x, y in zip(a, b))

ai_ctx  = dict((t, ctx) for t, c, ctx in FIXTURES if c == "ai")["diffusion"]
chem_ctx = dict((t, ctx) for t, c, ctx in FIXTURES if c == "chemistry")["diffusion"]

by_spelling = cosine(embed("diffusion"), embed("diffusion"))
by_meaning  = cosine(embed(ai_ctx), embed(chem_ctx))
control     = cosine(embed(ai_ctx), embed(
    "A denoising generative process that gradually removes noise to synthesise an image."))

print(f"same spelling, compared as spelling   : {by_spelling:.3f}   <- identical, merges them")
print(f"same spelling, compared as meaning    : {by_meaning:.3f}   <- separates them")
print(f"same meaning, different words         : {control:.3f}   <- still recognised as one sense")
print()
SEPARATION_THRESHOLD = 0.25   # calibrated for THIS crude vectoriser, not a universal constant
print(f"SEPARATION_THRESHOLD = {SEPARATION_THRESHOLD} — below this, two senses of one spelling are")
print("stored as distinct concepts with per-club meanings instead of merged into one node.")
print("verdict:", "SPLIT (homonym)" if by_meaning < SEPARATION_THRESHOLD else "MERGE (same sense)")
print()
print("The threshold is a property of the encoder, not of the idea. Swap in a real sentence")
print("encoder and both numbers rise — recalibrate it against known homonym pairs rather")
print("than carrying this value over.")

## 6 · Experiments

This is the part worth your time. Each cell turns exactly one dial.

### 6a · Re-weighting a lens

A maths club raises the Theorist. Watch which terms cross the 0.5 admission line, and in
which direction — the weight shifts the mean without ever giving one seat a veto.

In [ ]:
def with_weight(lens_key: str, w: float) -> List[Seat]:
    return [replace(s, weight=w) if s.key == lens_key else s for s in COUNCIL]

baseline = {(v.term, v.club): v for v in verdicts}

print(f"{'re-weighting':<26}{'mean delta rel':>16}   terms that cross the 0.5 line")
print("-" * 90)
for lens_key, w in [("theorist", 1.5), ("theorist", 2.0), ("visionary", 2.0),
                    ("skeptic", 1.5), ("skeptic", 2.0)]:
    out = [convene(t, c, club_focus(c), ctx, council=with_weight(lens_key, w))
           for t, c, ctx in FIXTURES]
    deltas, crossings = [], []
    for v in out:
        b = baseline[(v.term, v.club)]
        deltas.append(v.relevance - b.relevance)
        if v.kept != b.kept:
            crossings.append(f"{v.term}/{v.club} {b.relevance:.2f}->{v.relevance:.2f} "
                             f"{'ADMITTED' if v.kept else 'DROPPED'}")
    label = f"{lens_key} w={w}"
    print(f"{label:<26}{sum(deltas)/len(deltas):>+16.3f}   {', '.join(crossings) or '(none)'}")

print()
print("Weights move the mean; they never hand a seat a veto. A term only crosses the line")
print("when it was already sitting near it -- which is exactly the population a curator")
print("should be reviewing by hand anyway.")

### 6b · Removing a seat

How much of the verdict was one lens carrying? Disable each in turn and measure the mean
absolute shift. A lens whose removal barely moves anything is a lens you are paying for and
not using.

In [ ]:
print(f"{'seat removed':<20}{'mean |delta rel|':>18}{'verdict flips':>16}")
print("-" * 54)
for s in COUNCIL:
    council = [replace(x, enabled=False) if x.key == s.key else x for x in COUNCIL]
    out = [convene(t, c, club_focus(c), ctx, council=council) for t, c, ctx in FIXTURES]
    deltas = [abs(v.relevance - baseline[(v.term, v.club)].relevance) for v in out]
    flips = sum(1 for v in out if v.kept != baseline[(v.term, v.club)].kept)
    print(f"{s.persona[:19]:<20}{sum(deltas)/len(deltas):>18.3f}{flips:>16}")

### 6c · Temperature and repeatability

Production pins classification at `temperature=0.2` and paper ranking at `0.3`. This is the
cheapest possible argument for why: the same term, scored ten times, at each setting.

*(With the mock provider the variance is synthetic but faithful in shape — the jitter scales
with temperature exactly as sampling noise does. Set a real key to measure your model.)*

In [ ]:
term, club, ctx = FIXTURES[0]
print(f"{'temperature':>12}{'mean rel':>11}{'stdev':>9}{'min':>8}{'max':>8}")
print("-" * 48)
for temp in (0.0, 0.2, 0.5, 0.8, 1.0):
    council = [replace(s, temperature=temp) for s in COUNCIL]
    runs = []
    for i in range(10):
        # vary the context imperceptibly so the deterministic mock behaves like resampling
        v = convene(term, club, club_focus(club), ctx + " " * i, council=council)
        runs.append(v.relevance)
    print(f"{temp:>12.1f}{statistics.mean(runs):>11.3f}{statistics.pstdev(runs):>9.3f}"
          f"{min(runs):>8.3f}{max(runs):>8.3f}")

### 6d · Severity

Severity is not a score multiplier — it is the grading disposition handed to the seat, and it
reaches the model as an instruction, not as arithmetic. The failure mode it exists to fix is
a lens that is enthusiastic about everything, which in practice you hit far more often than a
lens that is too harsh.

In [ ]:
print(f"{'severity':>10}  {'instruction sent to the seat':<42}{'mean rel':>10}")
print("-" * 66)
for sev in (0.2, 0.5, 0.9):
    council = [replace(s, severity=sev) for s in COUNCIL]
    out = [convene(t, c, club_focus(c), ctx_, council=council) for t, c, ctx_ in FIXTURES]
    instr = ("Grade generously." if sev < 0.35 else
             "Grade as the rubric reads." if sev <= 0.65 else
             "Grade harshly: a 9 must be exceptional.")
    print(f"{sev:>10.1f}  {instr:<42}{statistics.mean(v.relevance for v in out):>10.3f}")

## 7 · Club presets

A council where every weight is `1.0` teaches the mechanism but not the craft. These are
reasoned starting points for the four live clubs — **not measured optima**. Each departure is
justified by the failure mode that lens prevents in that club's literature.

Two rules before you copy them:

1. **Tune the gate before the weights.** `keep_threshold` moves precision against recall
   directly and interpretably. Weights are a second-order correction on top of it.
2. **Only ratios matter.** Consensus is a weighted mean, so `2/1/1/1/1` and
   `1.0/0.5/0.5/0.5/0.5` are the same council. Weights need not sum to anything.

In production these live in `council_club_overrides` / `council_club_gates`, applied by
`set_council_club_override()` and resolved by `council_for_club(step, tier, club_id)`.

In [ ]:
PRESETS = {
    # club:      theorist engineer educator visionary skeptic  temp  gate   why
    "ai":        dict(w=(0.9, 1.2, 1.0, 0.8, 1.5), temp=0.20, gate=0.55,
                      why="Highest hype density + the field's own reproducibility problem. "
                          "Theorist DOWN: great AI work is often empirical and light on maths."),
    "maths":     dict(w=(1.4, 0.6, 1.3, 1.0, 0.7), temp=0.15, gate=0.50,
                      why="Rigour is native, so the Skeptic has less to do. The real gate is "
                          "teachability: correct is not the same as learnable."),
    "biomedical":dict(w=(0.8, 1.3, 1.1, 0.7, 1.6), temp=0.15, gate=0.60,
                      why="Overclaiming here is harm, not noise. Clinical timelines are the "
                          "most overclaimed quantity in the literature."),
    "physics":   dict(w=(1.3, 0.8, 1.0, 1.1, 1.0), temp=0.20, gate=0.50,
                      why="Ten domains, decade horizons. Much excellent physics is not "
                          "'usable tomorrow' and should not be marked down for it."),
    "default":   dict(w=(1.0, 1.0, 1.0, 1.0, 1.0), temp=0.20, gate=0.50,
                      why="Uniform prior — correct until you have review evidence."),
    "bulk-import":dict(w=(1.0, 1.0, 1.0, 0.8, 1.8), temp=0.15, gate=0.65,
                      why="Messy source: buy precision, pay for it in recall."),
}

ORDER = ["theorist", "engineer", "educator", "visionary", "skeptic"]

def apply_preset(name: str) -> List[Seat]:
    p = PRESETS[name]
    by_key = dict(zip(ORDER, p["w"]))
    return [replace(s, weight=by_key[s.key], temperature=p["temp"]) for s in COUNCIL]

print(f"{'club':<13}" + "".join(f"{k[:5]:>8}" for k in ORDER) + f"{'temp':>7}{'gate':>7}")
print("-" * 68)
for name, p in PRESETS.items():
    print(f"{name:<13}" + "".join(f"{w:>8.1f}" for w in p["w"]) + f"{p['temp']:>7.2f}{p['gate']:>7.2f}")

### The presets, applied

Same seven fixtures, judged by each club's preset. The point is not the absolute numbers —
the mock provider makes those synthetic — but that the presets **disagree with each other**,
and that the disagreement is concentrated on the borderline terms. A preset that changes
nothing is a preset you do not need.

In [ ]:
rows = {}
for name in PRESETS:
    council = apply_preset(name)
    gate = PRESETS[name]["gate"]
    out = []
    for t, c, ctx in FIXTURES:
        v = convene(t, c, club_focus(c), ctx, council=council)
        out.append((v, v.relevance >= gate))     # this preset's gate, not the global one
    rows[name] = out

print(f"{'term / club':<28}" + "".join(f"{n[:11]:>12}" for n in PRESETS))
print("-" * (28 + 12 * len(PRESETS)))
for i, (t, c, _) in enumerate(FIXTURES):
    line = f"{(t + ' / ' + c)[:27]:<28}"
    for name in PRESETS:
        v, kept = rows[name][i]
        line += f"{v.relevance:>9.2f}{'*' if kept else ' ':<3}"
    print(line)
print("\n* = admitted under that preset's own gate")

print("\nHow much does the preset choice matter?")
print(f"{'preset':<13}{'admitted':>10}{'mean rel':>11}   why")
print("-" * 100)
for name, p in PRESETS.items():
    kept = sum(1 for _, k in rows[name] if k)
    mean = statistics.mean(v.relevance for v, _ in rows[name])
    print(f"{name:<13}{kept:>10}{mean:>11.3f}   {p['why'][:60]}")

### Is a preset wrong?

Three signals, all cheaper than intuition. Run these against **your** data, not the fixtures.

> **Reading this under the mock provider.** The mock's five lenses are deliberately similar to
> each other, so signal 2 will flag almost every seat as "not voting". That is a property of
> the mock, not a finding about your council. Signal 2 only means something against a real
> model, where the lenses genuinely diverge. Signals 1 and 3 are informative either way.

In [ ]:
def diagnose(council: List[Seat], gate: float, fixtures=FIXTURES) -> None:
    base = [convene(t, c, club_focus(c), ctx, council=council) for t, c, ctx in fixtures]
    base_kept = [v.relevance >= gate for v in base]

    print("1. Is any weight doing the GATE's job? (bulk crossings on a small nudge)")
    for s in council:
        bumped = [replace(x, weight=x.weight * 1.25) if x.key == s.key else x for x in council]
        out = [convene(t, c, club_focus(c), ctx, council=bumped) for t, c, ctx in fixtures]
        flips = sum(1 for v, was in zip(out, base_kept) if (v.relevance >= gate) != was)
        warn = "  <-- weight is gating" if flips > len(fixtures) * 0.25 else ""
        print(f"   {s.persona:<18} +25% weight -> {flips} verdict flip(s){warn}")

    print("\n2. Is any seat not voting? (mean |delta| under 0.02 = you are paying for nothing)")
    for s in council:
        without = [replace(x, enabled=False) if x.key == s.key else x for x in council]
        out = [convene(t, c, club_focus(c), ctx, council=without) for t, c, ctx in fixtures]
        d = statistics.mean(abs(v.relevance - b.relevance) for v, b in zip(out, base))
        warn = "  <-- retire it or sharpen its rubric" if d < 0.02 else ""
        print(f"   {s.persona:<18} removed -> mean |delta rel| {d:.3f}{warn}")

    print("\n3. Do the lenses disagree structurally? (spread > 4 on ADMITTED terms)")
    admitted = [v for v, k in zip(base, base_kept) if k]
    if not admitted:
        print("   (nothing admitted at this gate — lower it before reading this signal)")
    else:
        wide = [v for v in admitted if v.spread > 4.0]
        print(f"   {len(wide)}/{len(admitted)} admitted terms have spread > 4"
              f"   (mean spread {statistics.mean(v.spread for v in admitted):.2f})")
        if wide:
            print("   -> rubric problem, not a weight problem. Re-weighting will not fix it.")

print("=== diagnosing the 'ai' preset ===")
diagnose(apply_preset("ai"), PRESETS["ai"]["gate"])

## 8 · One model or five? A measured answer

The production knowledge-graph council runs **one API call per batch** — five personas and
thirty terms share a single completion (`scripts/classify-term-clubs.mjs:254`). That buys
determinism and cost, and it raises the obvious question: what does the council actually lose
by not giving each seat its own model?

This section measures it on the homonym the blog opens with. Three conditions:

| | Calls | Models | Isolates |
|---|---|---|---|
| **A** single-call | 1 | 1 | production today |
| **B** per-seat, same model | 5 | 1 × 5 | the effect of **splitting the call** |
| **C** per-seat, diverse | 5 | 5 distinct | the effect of **model diversity** |

A → B changes only the call structure. B → C changes only the models. Four cases: each sense
of *diffusion* judged for its own club, and each judged for the **other** club — the two
cross-club traps a knowledge graph must reject. Two tiers (quorum = 3 seats, full = 5),
three repeats per cell.

In [ ]:
# Multi-provider client. Fill in whichever keys you have — the experiment runs with any two
# distinct models, and is most informative across DIFFERENT VENDORS (different training
# lineages), which is the actual argument for a council.
import urllib.request, json, os, re, statistics, time
from concurrent.futures import ThreadPoolExecutor

PROVIDERS = {
    "anthropic": dict(url="https://api.anthropic.com/v1/messages",
                      key=os.environ.get("ANTHROPIC_API_KEY", "")),
    "openai":    dict(url="https://api.openai.com/v1/chat/completions",
                      key=os.environ.get("OPENAI_API_KEY", "")),
    "deepseek":  dict(url="https://api.deepseek.com/chat/completions",
                      key=os.environ.get("DEEPSEEK_API_KEY", "")),
    "gemini":    dict(url="https://generativelanguage.googleapis.com/v1beta/models",
                      key=os.environ.get("GEMINI_API_KEY", "")),
}

def llm(provider: str, model: str, prompt: str, temperature: float = 0.2) -> str:
    p = PROVIDERS[provider]
    if not p["key"]:
        raise RuntimeError(f"no key for {provider}")
    if provider == "anthropic":
        url, hdr = p["url"], {"x-api-key": p["key"], "anthropic-version": "2023-06-01",
                              "content-type": "application/json"}
        body = {"model": model, "max_tokens": 700, "temperature": temperature,
                "messages": [{"role": "user", "content": prompt}]}
        pick = lambda r: "".join(b.get("text", "") for b in r.get("content", []))
    elif provider == "gemini":
        url = f"{p['url']}/{model}:generateContent?key={p['key']}"
        hdr = {"content-type": "application/json"}
        body = {"contents": [{"parts": [{"text": prompt}]}],
                "generationConfig": {"temperature": temperature, "responseMimeType": "application/json"}}
        pick = lambda r: r["candidates"][0]["content"]["parts"][0]["text"]
    else:   # openai-compatible: openai, deepseek
        url, hdr = p["url"], {"Content-Type": "application/json",
                              "Authorization": f"Bearer {p['key']}"}
        body = {"model": model, "messages": [{"role": "user", "content": prompt}],
                "temperature": temperature, "max_tokens": 700,
                "response_format": {"type": "json_object"}}
        pick = lambda r: r["choices"][0]["message"]["content"]

    req = urllib.request.Request(url, data=json.dumps(body).encode(), headers=hdr)
    last = None
    for attempt in range(4):
        try:
            with urllib.request.urlopen(req, timeout=120) as r:
                return pick(json.loads(r.read()))
        except Exception as e:      # noqa: BLE001
            last = e; time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f"{provider}/{model}: {last}")

def parse_json(text):
    t = re.sub(r"^\s*```(?:json)?|```\s*$", "", text.strip())
    try:
        return json.loads(t)
    except Exception:               # noqa: BLE001
        a, b = t.find("{"), t.rfind("}")
        return json.loads(t[a:b + 1])

# EDIT ME — the whole point of the experiment is that these are different lineages.
SAME_MODEL = ("anthropic", "claude-sonnet-4-5-20250929")
DIVERSE = {                          # ideally one vendor per seat
    "theorist":  ("anthropic", "claude-opus-4-5-20251101"),
    "engineer":  ("openai",    "gpt-4o"),
    "skeptic":   ("deepseek",  "deepseek-chat"),
    "educator":  ("gemini",    "gemini-2.5-flash"),
    "visionary": ("anthropic", "claude-opus-4-1-20250805"),
}
QUORUM_LENSES = ["theorist", "engineer", "skeptic"]

SENSES = {
  "ai": "A generative model that starts from pure Gaussian noise and iteratively denoises it "
        "into an image, learning to reverse a forward noising process.",
  "physics": "The net transport of particles from a region of higher concentration to lower "
             "concentration, driven by the concentration gradient and random thermal motion.",
}
CLUB_DESC = {
  "ai": "Artificial Intelligence Collective. Machine learning, deep learning, generative "
        "models, NLP, computer vision, reinforcement learning, AI theory and safety.",
  "physics": "Physics Scholars Collective. Condensed matter, statistical mechanics, "
             "thermodynamics, particle physics, cosmology, molecular transport phenomena.",
}
CASES = [("ai-sense -> ai club","ai","ai",True),
         ("physics-sense -> physics club","physics","physics",True),
         ("ai-sense -> physics club","ai","physics",False),          # trap
         ("physics-sense -> ai club","physics","ai",False)]          # trap

def seat_prompt(lens, sense, club):
    seat = next(s for s in COUNCIL if s.key == lens)
    return f"""{seat.rubric}

You are one member of the "Jedi Council" of a learning club — a panel judging whether a
concept genuinely belongs to this club's field.

CLUB: {CLUB_DESC[club]}

TERM: diffusion

MEANING AS USED IN THE SOURCE PAPERS: {SENSES[sense]}

Judge ONLY through your own lens. Return STRICT JSON, no prose:
{{"score": <0-10>, "is_home": <true|false>, "rationale": "<one sentence>"}}"""

def panel_prompt(lenses, sense, club):
    roster = "\n".join(f"- THE {l.upper()}: "
                       f"{next(s for s in COUNCIL if s.key==l).rubric}" for l in lenses)
    keys = ", ".join(f'"{l}": <0-10>' for l in lenses)
    return f"""You are the "Jedi Council" of a learning club — {len(lenses)} distinct expert
evaluators judging whether a concept belongs to this club's field.

{roster}

CLUB: {CLUB_DESC[club]}

TERM: diffusion

MEANING AS USED IN THE SOURCE PAPERS: {SENSES[sense]}

Have every member score it on their own lens, then agree a consensus.
Return STRICT JSON, no prose:
{{{keys}, "relevance": <0.0-1.0>, "is_home": <true|false>, "rationale": "<one sentence>"}}"""

def run_condition(condition, tier, case, repeats=3):
    _, sense, club, _ = case
    lenses = QUORUM_LENSES if tier == "quorum" else [s.key for s in COUNCIL]
    out = []
    for _ in range(repeats):
        if condition == "A":
            d = parse_json(llm(*SAME_MODEL, panel_prompt(lenses, sense, club)))
            scores = {l: float(d.get(l, 0)) for l in lenses}
            rel = float(d.get("relevance", statistics.mean(scores.values()) / 10))
            home = bool(d.get("is_home"))
        else:
            models = {l: (SAME_MODEL if condition == "B" else DIVERSE[l]) for l in lenses}
            with ThreadPoolExecutor(max_workers=5) as ex:
                got = list(ex.map(lambda l: (l, parse_json(
                    llm(*models[l], seat_prompt(l, sense, club)))), lenses))
            scores = {l: float(o.get("score", 0)) for l, o in got}
            rel = statistics.mean(scores.values()) / 10
            home = sum(bool(o.get("is_home")) for _, o in got) > len(got) / 2
        out.append(dict(scores=scores, relevance=rel, is_home=home))
    return out

print("Harness ready. `run_condition('C','full',CASES[3])` runs one cell "
      "(costs real tokens).\nRecorded results from a previous run are analysed below, "
      "so the rest of this section works with no keys.")

### Recorded run

Baked in from an actual execution so this section is readable and reproducible without
spending anything. **Caveat, and it matters:** only one vendor was reachable from the machine
that ran it, so condition C varied *model size and generation within one family* rather than
across training lineages. Cross-lineage diversity — the real argument for a council — should
show these effects **more** strongly, not less. Re-run the harness above with your own four
providers to measure that properly.

Setup: temperature 0.2, 3 repeats per cell, 216 calls total.

In [ ]:
RECORDED = json.loads(r"""[
 {
  "condition": "A",
  "tier": "quorum",
  "case": "ai-sense -> ai club",
  "per_lens": {
   "theorist": 9.0,
   "engineer": 9.0,
   "skeptic": 9.0
  },
  "rel_mean": 1.0,
  "rel_sd": 0.0,
  "spread_mean": 0.0,
  "is_home_rate": 1.0
 },
 {
  "condition": "A",
  "tier": "quorum",
  "case": "physics-sense -> physics club",
  "per_lens": {
   "theorist": 10.0,
   "engineer": 10.0,
   "skeptic": 10.0
  },
  "rel_mean": 1.0,
  "rel_sd": 0.0,
  "spread_mean": 0.0,
  "is_home_rate": 1.0
 },
 {
  "condition": "A",
  "tier": "quorum",
  "case": "ai-sense -> physics club",
  "per_lens": {
   "theorist": 2.0,
   "engineer": 1.0,
   "skeptic": 1.0
  },
  "rel_mean": 0.1,
  "rel_sd": 0.0,
  "spread_mean": 1.0,
  "is_home_rate": 0.0
 },
 {
  "condition": "A",
  "tier": "quorum",
  "case": "physics-sense -> ai club",
  "per_lens": {
   "theorist": 9.0,
   "engineer": 9.0,
   "skeptic": 9.0
  },
  "rel_mean": 1.0,
  "rel_sd": 0.0,
  "spread_mean": 0.0,
  "is_home_rate": 1.0
 },
 {
  "condition": "A",
  "tier": "full",
  "case": "ai-sense -> ai club",
  "per_lens": {
   "theorist": 9.333333333333334,
   "engineer": 10.0,
   "educator": 7.333333333333333,
   "visionary": 10.0,
   "skeptic": 9.0
  },
  "rel_mean": 1.0,
  "rel_sd": 0.0,
  "spread_mean": 2.6666666666666665,
  "is_home_rate": 1.0
 },
 {
  "condition": "A",
  "tier": "full",
  "case": "physics-sense -> physics club",
  "per_lens": {
   "theorist": 10.0,
   "engineer": 10.0,
   "educator": 10.0,
   "visionary": 9.0,
   "skeptic": 10.0
  },
  "rel_mean": 1.0,
  "rel_sd": 0.0,
  "spread_mean": 1.0,
  "is_home_rate": 1.0
 },
 {
  "condition": "A",
  "tier": "full",
  "case": "ai-sense -> physics club",
  "per_lens": {
   "theorist": 2.0,
   "engineer": 1.0,
   "educator": 3.0,
   "visionary": 1.0,
   "skeptic": 2.0
  },
  "rel_mean": 0.13333333333333333,
  "rel_sd": 0.02357022603955158,
  "spread_mean": 2.0,
  "is_home_rate": 0.0
 },
 {
  "condition": "A",
  "tier": "full",
  "case": "physics-sense -> ai club",
  "per_lens": {
   "theorist": 9.0,
   "engineer": 9.0,
   "educator": 8.0,
   "visionary": 10.0,
   "skeptic": 9.0
  },
  "rel_mean": 1.0,
  "rel_sd": 0.0,
  "spread_mean": 2.0,
  "is_home_rate": 1.0
 },
 {
  "condition": "B",
  "tier": "quorum",
  "case": "ai-sense -> ai club",
  "per_lens": {
   "theorist": 9.0,
   "engineer": 9.0,
   "skeptic": 10.0
  },
  "rel_mean": 0.9333333333333333,
  "rel_sd": 0.0,
  "spread_mean": 1.0,
  "is_home_rate": 1.0
 },
 {
  "condition": "B",
  "tier": "quorum",
  "case": "physics-sense -> physics club",
  "per_lens": {
   "theorist": 9.0,
   "engineer": 10.0,
   "skeptic": 10.0
  },
  "rel_mean": 0.9666666666666666,
  "rel_sd": 0.0,
  "spread_mean": 1.0,
  "is_home_rate": 1.0
 },
 {
  "condition": "B",
  "tier": "quorum",
  "case": "ai-sense -> physics club",
  "per_lens": {
   "theorist": 3.0,
   "engineer": 0.0,
   "skeptic": 2.3333333333333335
  },
  "rel_mean": 0.17777777777777778,
  "rel_sd": 0.01571348402636772,
  "spread_mean": 3.0,
  "is_home_rate": 0.0
 },
 {
  "condition": "B",
  "tier": "quorum",
  "case": "physics-sense -> ai club",
  "per_lens": {
   "theorist": 9.0,
   "engineer": 8.0,
   "skeptic": 3.0
  },
  "rel_mean": 0.6666666666666667,
  "rel_sd": 0.0,
  "spread_mean": 6.0,
  "is_home_rate": 1.0
 },
 {
  "condition": "B",
  "tier": "full",
  "case": "ai-sense -> ai club",
  "per_lens": {
   "theorist": 9.0,
   "engineer": 9.0,
   "educator": 7.0,
   "visionary": 10.0,
   "skeptic": 10.0
  },
  "rel_mean": 0.9,
  "rel_sd": 0.0,
  "spread_mean": 3.0,
  "is_home_rate": 1.0
 },
 {
  "condition": "B",
  "tier": "full",
  "case": "physics-sense -> physics club",
  "per_lens": {
   "theorist": 9.0,
   "engineer": 10.0,
   "educator": 10.0,
   "visionary": 10.0,
   "skeptic": 10.0
  },
  "rel_mean": 0.9800000000000001,
  "rel_sd": 0.0,
  "spread_mean": 1.0,
  "is_home_rate": 1.0
 },
 {
  "condition": "B",
  "tier": "full",
  "case": "ai-sense -> physics club",
  "per_lens": {
   "theorist": 3.0,
   "engineer": 0.0,
   "educator": 2.0,
   "visionary": 3.0,
   "skeptic": 2.6666666666666665
  },
  "rel_mean": 0.21333333333333335,
  "rel_sd": 0.009428090415820642,
  "spread_mean": 3.0,
  "is_home_rate": 0.0
 },
 {
  "condition": "B",
  "tier": "full",
  "case": "physics-sense -> ai club",
  "per_lens": {
   "theorist": 9.0,
   "engineer": 8.0,
   "educator": 8.666666666666666,
   "visionary": 10.0,
   "skeptic": 3.0
  },
  "rel_mean": 0.7733333333333333,
  "rel_sd": 0.009428090415820642,
  "spread_mean": 7.0,
  "is_home_rate": 1.0
 },
 {
  "condition": "C",
  "tier": "quorum",
  "case": "ai-sense -> ai club",
  "per_lens": {
   "theorist": 8.0,
   "engineer": 9.0,
   "skeptic": 9.0
  },
  "rel_mean": 0.8666666666666666,
  "rel_sd": 0.0,
  "spread_mean": 1.0,
  "is_home_rate": 1.0
 },
 {
  "condition": "C",
  "tier": "quorum",
  "case": "physics-sense -> physics club",
  "per_lens": {
   "theorist": 8.0,
   "engineer": 10.0,
   "skeptic": 9.0
  },
  "rel_mean": 0.9,
  "rel_sd": 0.0,
  "spread_mean": 2.0,
  "is_home_rate": 1.0
 },
 {
  "condition": "C",
  "tier": "quorum",
  "case": "ai-sense -> physics club",
  "per_lens": {
   "theorist": 2.0,
   "engineer": 0.0,
   "skeptic": 2.0
  },
  "rel_mean": 0.13333333333333333,
  "rel_sd": 0.0,
  "spread_mean": 2.0,
  "is_home_rate": 0.0
 },
 {
  "condition": "C",
  "tier": "quorum",
  "case": "physics-sense -> ai club",
  "per_lens": {
   "theorist": 2.0,
   "engineer": 8.0,
   "skeptic": 9.0
  },
  "rel_mean": 0.6333333333333333,
  "rel_sd": 0.0,
  "spread_mean": 7.0,
  "is_home_rate": 1.0
 },
 {
  "condition": "C",
  "tier": "full",
  "case": "ai-sense -> ai club",
  "per_lens": {
   "theorist": 8.0,
   "engineer": 9.0,
   "educator": 9.0,
   "visionary": 9.0,
   "skeptic": 9.0
  },
  "rel_mean": 0.8800000000000001,
  "rel_sd": 0.0,
  "spread_mean": 1.0,
  "is_home_rate": 1.0
 },
 {
  "condition": "C",
  "tier": "full",
  "case": "physics-sense -> physics club",
  "per_lens": {
   "theorist": 8.0,
   "engineer": 10.0,
   "educator": 9.0,
   "visionary": 8.0,
   "skeptic": 9.0
  },
  "rel_mean": 0.8800000000000001,
  "rel_sd": 0.0,
  "spread_mean": 2.0,
  "is_home_rate": 1.0
 },
 {
  "condition": "C",
  "tier": "full",
  "case": "ai-sense -> physics club",
  "per_lens": {
   "theorist": 2.0,
   "engineer": 0.0,
   "educator": 8.0,
   "visionary": 7.0,
   "skeptic": 2.0
  },
  "rel_mean": 0.38,
  "rel_sd": 0.0,
  "spread_mean": 8.0,
  "is_home_rate": 0.0
 },
 {
  "condition": "C",
  "tier": "full",
  "case": "physics-sense -> ai club",
  "per_lens": {
   "theorist": 2.0,
   "engineer": 8.0,
   "educator": 6.0,
   "visionary": 9.0,
   "skeptic": 9.0
  },
  "rel_mean": 0.6799999999999999,
  "rel_sd": 0.0,
  "spread_mean": 7.0,
  "is_home_rate": 1.0
 }
]""")

def cell(c, t, case_fragment):
    return next(r for r in RECORDED
                if r["condition"] == c and r["tier"] == t and case_fragment in r["case"])

NAMES = {"A": "A single-call, 1 model", "B": "B per-seat, 1 model", "C": "C per-seat, 5 models"}

print("DISCRIMINATION — on-target relevance vs the cross-club trap (bigger gap = cleaner separation)")
print(f"{'condition':<24}{'tier':<8}{'on-target':>11}{'trap':>8}{'gap':>8}")
print("-" * 59)
for c in ("A", "B", "C"):
    for t in ("quorum", "full"):
        on = statistics.mean([cell(c,t,"ai-sense -> ai club")["rel_mean"],
                              cell(c,t,"physics-sense -> physics club")["rel_mean"]])
        tr = statistics.mean([cell(c,t,"ai-sense -> physics club")["rel_mean"],
                              cell(c,t,"physics-sense -> ai club")["rel_mean"]])
        print(f"{NAMES[c]:<24}{t:<8}{on:>11.3f}{tr:>8.3f}{on-tr:>8.3f}")

print("\n\nDISAGREEMENT SIGNAL — mean spread between lenses, and run-to-run stdev")
print(f"{'condition':<24}{'tier':<8}{'lens spread':>13}{'run-to-run sd':>15}")
print("-" * 60)
for c in ("A", "B", "C"):
    for t in ("quorum", "full"):
        rs = [r for r in RECORDED if r["condition"] == c and r["tier"] == t]
        print(f"{NAMES[c]:<24}{t:<8}"
              f"{statistics.mean(r['spread_mean'] for r in rs):>13.2f}"
              f"{statistics.mean(r['rel_sd'] for r in rs):>15.3f}")

### The one that matters

Look at the trap every condition got **wrong** — molecular diffusion offered to the AI club.
All three admitted it (`is_home` true in 100% of runs). Model diversity did not fix the
verdict. What changed is whether the panel could *say it was unsure*.

In [ ]:
print("physics-sense (molecular diffusion) judged for the AI club — per-lens means\n")
cols = ["theorist", "engineer", "skeptic", "educator", "visionary"]
print(f"{'condition':<24}{'tier':<8}" + "".join(f"{c[:5]:>8}" for c in cols)
      + f"{'spread':>9}{'rel':>7}{'admitted':>10}")
print("-" * 96)
for c in ("A", "B", "C"):
    for t in ("quorum", "full"):
        r = cell(c, t, "physics-sense -> ai club")
        row = f"{NAMES[c]:<24}{t:<8}"
        for l in cols:
            row += f"{r['per_lens'][l]:>8.1f}" if l in r["per_lens"] else f"{'-':>8}"
        print(row + f"{r['spread_mean']:>9.1f}{r['rel_mean']:>7.2f}{r['is_home_rate']:>9.0%}")

print("""
READ IT LIKE THIS

A / quorum — three personas, one call, one model: 9.0, 9.0, 9.0. Spread 0.0. The panel is
  unanimous and unanimously wrong. This is not a council; it is one model asked to
  role-play three, and it agrees with itself for free.

B / quorum — same model, five separate calls: the Skeptic breaks ranks at 3.0 while the
  Theorist holds 9.0. Spread 6.0. Splitting the call recovers most of the doubt signal
  WITHOUT changing a single model.

C / quorum — different model per seat: now the THEORIST dissents, hard, at 2.0. A different
  seat objects, for a different reason, than under B. Spread 7.0.

The consensus barely moves (1.00 -> 0.67 -> 0.63). The spread moves from 0.0 to 7.0.
So the thing model diversity buys is not a better mean. It is a mean you can DISTRUST.
""")

### What to do with that

Three conclusions, in the order they cost you effort:

1. **Splitting the call is where most of the value is.** A → B recovered spread 0.0 → 6.0 with
   the model held constant. Five personas in one prompt cannot disagree with themselves in any
   meaningful way, because one forward pass produces one point of view wearing five hats. If
   you change one thing, change this.

2. **Model diversity changes *who* dissents, not *whether* the mean is right.** Under B the
   Skeptic objected; under C the Theorist did, and harder. Different lineages fail
   differently — which is the entire argument for a council, and also why the fix for a wrong
   verdict is rarely "add another judge".

3. **Route on spread, not on the mean.** Every condition admitted the trap on consensus alone.
   Only the spread distinguished the confident-correct cases (spread ≤ 2) from the
   confident-wrong one (spread 6-7). A cheap, deployable rule:

   ```
   if relevance >= gate and spread <= 3:   admit
   if relevance >= gate and spread >  3:   admit, flag for human review
   if relevance <  gate and spread >  3:   hold — the council is split, not sure
   else:                                   drop
   ```

   That is a change to `classify-term-clubs.mjs` worth more than any weight in section 7,
   and it costs nothing extra to compute: the spread is already sitting in the per-lens scores
   the job throws away.

One honest caveat on the fourth case. Molecular diffusion is not an absurd thing to find in an
AI club — diffusion-on-graphs, SDE formulations of generative models, and Brownian motion all
genuinely connect the two. The models were not hallucinating a link; they were weighing a real
one. Which is precisely why the *spread* is the signal worth acting on: it separates "this is
clearly home" from "there is an argument here", and only a human should settle the second.

## 9 · How this maps to production

| Notebook | Production |
|---|---|
| `Seat` dataclass | `public.ai_council_members` — one row per (step, seat), edited via the admin-gated `set_council_member()` RPC, audited to `admin_audit_log` |
| `Seat.rubric` | `public.prompt_registry`, keys `council/lens-*` — append-only versions via `save_prompt()` |
| `get_provider()` | `public.ai_model_registry` — which models may take a seat, with modalities, context window and price per Mtok |
| `convene()` | `scripts/classify-term-clubs.mjs` (temp 0.2) and `scripts/select-weekly-drop.mjs` (temp 0.3) |
| `KEEP_THRESHOLD` | `relevance >= 0.5` gate; verdicts land in `term_club_relevance` |
| `PRESETS` / `apply_preset()` | `public.council_club_overrides` + `public.council_club_gates`, resolved by `council_for_club(step, tier, club_id)` |
| `embed()` / `cosine()` | contextual embeddings + per-club `sense_label`, `hook`, `feynman_body`, verified by a second model |

**One honest difference.** This notebook makes one model call per seat. The production
knowledge-graph council currently runs the five lenses as five personas inside a *single*
call — cheaper, faster, and determinstic enough to classify thousands of terms, at the cost
of five personas sharing one model's blind spots. The genuinely per-seat, multi-model path is
live on the Urdu pipeline; that is
[Part 2](https://ilm.red/blog/the-jedi-council-part-2-teaching-ai-to-read-urdu-aloud).

Running this notebook with a real key and one call per seat is therefore not a simulation of
production — it is the *upgrade* to production, and the numbers you get are the evidence for
whether the extra four calls are worth their cost on your data.

**Found something that contradicts the blog?** Leave a review on
[the post](https://ilm.red/blog/meet-the-ai-jedi-council-keeping-a-machine-built-knowledge-graph-honest) —
a reproducible disagreement is the most useful thing anyone can send us.